# Zero-to-Hero: Production Semantic Search Engine\n\nThis notebook walks through embeddings, chunking, vector indexing, hybrid retrieval, reranking, and evaluation for Project #19.

## 1) Setup\n\nUsing `uv`, Python 3.12.10, ChromaDB, FAISS, Sentence Transformers, and Streamlit app integration.

In [ ]:
from semantic_search.config import load_config\nfrom semantic_search.schemas import SearchRequest\nfrom semantic_search.service import SemanticSearchService\n\nconfig = load_config('config/default.yaml')\nservice = SemanticSearchService(config)

## 2) Dataset Ingestion\n\nLoad 50k sampled documents from `khalidalt/HuffPost` and optionally enrich with full text from source URLs.

In [ ]:
service.ensure_ollama_models(include_optional_qwen=False)\ndocs = service.ingest_huggingface()\nlen(docs)

## 3) Chunking Strategies\n\nCompare recursive/token/semantic chunking across size and overlap settings.

In [ ]:
chunks = service.chunk_documents(strategy='recursive', chunk_size=512, chunk_overlap=50)\nlen(chunks)

## 4) Build Index + Retrieve\n\nBuild ChromaDB and FAISS indexes, run hybrid search, and inspect metadata-aware ranked hits.

In [ ]:
service.build_indexes(config.embedding.primary)\nresponse = service.search(SearchRequest(query='climate policy updates', mode='hybrid', top_k=5, rerank=True))

## 5) Evaluation Metrics\n\nCompute Precision@K, Recall@K, MRR, NDCG and latency from labeled evaluation set.

In [ ]:
# Generate and evaluate weak labels first\n# !python scripts/generate_eval_queries.py\neval_output = service.evaluate(\n    cases_path='data/processed/evaluation_queries.jsonl',\n    output_path='artifacts/reports/evaluation_results.json',\n    mode='hybrid',\n)\neval_output.summary

## 6) Visual Analytics\n\nUse PCA/t-SNE/UMAP projections and latency/quality plots from benchmark outputs.